# C3 · Extracción óptima

**Spec:** [`docs/spec_C3_codex_optimal_extraction.md`](../docs/spec_C3_codex_optimal_extraction.md)  |  **Bloque:** C · Extracción  |  **Run por defecto:** `ROXs12b_realigned`

Extracción óptima (Horne) ponderada por la PSF.

| | |
|---|---|
| **Entrada** | Cubo + PSF (C1) |
| **Salida (QC/productos)** | `stages/spec_optimal_qc.json` |
| **Consume aguas abajo** | D1, E1 |


## Qué hace C3 y las dos variantes

C3 es **extracción óptima de Horne (1986)**: por canal, pondera cada píxel por el **perfil de PSF esperado** (de C1) y la varianza inversa (`f = Σ M·P·D/V / Σ M·P²/V`). Al bajar el peso de los píxeles ruidosos, **gana S/N** frente a la apertura (aquí ~**6.9× mediana**). La fórmula es cerrada; el valor está en implementarla exacta (tests analíticos de flujo y varianza).

**Dos variantes del fondo:**
- **`optimal_ls`** — usa el residual de superficie local (04b) como fondo.
- **`optimal_psfsub`** — ajusta y **resta la PSF de la primaria** primero, y luego extrae ópticamente el compañero.

**Decisión:** G1 **valida `psfsub`** (`validated_with_bias`) y la usa como una de las dos citables (con psffit). **`ls` sobre-sustrae el continuo** (el pedestal de 04b) → sesgo de continuo **−373 % vs apertura**, `v3_continuum_bias` **falla**. Ambas comparten la forma roja real (SED de enana fría) pero `ls` queda con un gran desplazamiento negativo.

Errores **empíricos** (M5 rojo); **robusta a errores de PSF** (±10 % FWHM → 0 % de sesgo de flujo).


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x02_optimal.sh --run-id $RUN
```

Moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x02_optimal.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/spec_optimal_qc.json', RUN_ID)
nb.show(qc, keys=['variants', 'snr_gain_vs_aperture.median', 'continuum_bias_vs_aperture_pct', 'v3_continuum_bias_ok', 'fwhm_pm10pct'], title='C3')


## Resultados que llevaron a la conclusión

Ganancia de S/N, sesgo de continuo, modelo psfsub y chequeos del `spec_optimal_qc.json`.


In [ ]:
q = nb.load_qc('stages/spec_optimal_qc.json', RUN_ID)
sg = q['snr_gain_vs_aperture']; ck = q['checks']; pm = q['psfsub_model']
print('variantes:', q['variants'], '| ventana:', q['window_px'], 'px | PSF:', q['psf_model'].split('/')[-1])
print(f"ganancia S/N vs apertura: mediana {sg['median']:.2f}× (p10 {sg['p10']:.2f}, p90 {sg['p90']:.1f})")
print(f"sesgo de continuo vs apertura: {q['continuum_bias_vs_aperture_pct']:.0f}%   <-- ls sobre-sustrae")
print(f"sensibilidad a PSF (±10% FWHM): {q['psf_sensitivity']['fwhm_pm10pct_flux_bias_pct']:.1f}% de sesgo (robusto)")
print(f"psfsub: amplitud mediana {pm['amplitude_median']:.3g}, n_fit {pm['n_fit_median']:.0f}, "
      f"fit_radius {pm['fit_radius_px']:.0f}px, exclude {pm['exclude_radius_px']:.0f}px")
print(f"checks: v1_snr_gain={ck['v1_snr_gain_ok']} v2_error={ck['v2_error_ratio_ok']} "
      f"v3_continuum_bias={ck['v3_continuum_bias_ok']} (falla: sesgo ls) v4_clip={ck['v4_clip_concentration_ok']}")
print('open_issue:', q['open_issues'][0])


## Plot — las dos variantes: ls vs psfsub

Los espectros suavizados de `spec_optimal_object.fits` (ls) y `spec_optimal_psfsub_object.fits` (psfsub). **`ls` (naranja)** queda muy por debajo de cero = sobre-sustracción del continuo (el sesgo −373 %); **`psfsub` (verde)** queda cerca de cero y **sube al rojo** (SED real de enana fría). Comparten la forma, difieren en nivel — por eso G1 valida psfsub y rechaza ls.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    def spec(path):
        h = fits.open(rd / 'stages' / path); d = h[1].data
        w = np.asarray(d['wave_A'], float); f = np.asarray(d['flux'], float); h.close(); return w, f
    sm = lambda x, n=41: np.convolve(np.nan_to_num(x), np.ones(n) / n, mode='same')
    w, fls = spec('spec_optimal_object.fits')
    _, fps = spec('spec_optimal_psfsub_object.fits')
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(w, sm(fls), lw=1.2, color='tab:orange', label='optimal_ls (superficie local)')
    ax.plot(w, sm(fps), lw=1.2, color='tab:green', label='optimal_psfsub (resta de PSF) — validada G1')
    ax.axhline(0, color='0.6', lw=0.7); ax.axvline(6563, color='tab:red', ls=':', label='Hα')
    allv = np.concatenate([sm(fls), sm(fps)])
    ax.set_ylim(np.nanpercentile(allv, 2), np.nanpercentile(allv, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (suavizado 41ch)')
    ax.set_title('C3 · dos variantes: ls sobre-sustrae el continuo (−373% vs apertura), psfsub no')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c3_optimal'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'ls_vs_psfsub.png', dpi=110); print('figura ->', outdir / 'ls_vs_psfsub.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Extracción óptima de Horne ponderada por la PSF de C1 → **~6.9× ganancia de S/N** vs apertura (`v1` pasa).
- Dos variantes: **`optimal_psfsub` validada por G1** (`validated_with_bias`) y **`optimal_ls` rechazada** — sobre-sustrae el continuo (sesgo −373%, `v3` falla).
- Errores empíricos (M5 rojo); robusta a errores de PSF (±10% FWHM → 0% de sesgo de flujo).


## Conclusión (registrada)

**C3: extracción óptima de Horne; ~6.9× ganancia de S/N vs apertura; dos variantes (ls, psfsub).**

- **Fecha:** cadena D1 v2 sobre el run realineado (2026-07-09).
- **Entrada:** cubo stage02 + PSF de C1 (`psf_model.json`); ventana 8 px.
- **psfsub** (resta de PSF de la primaria): validada por G1, una de las dos citables.
- **ls** (superficie local): **sobre-sustrae** el continuo (−373 % vs apertura, `v3_continuum_bias` falla) — rechazada.
- **Robusta a PSF** (±10 % FWHM → 0 % de sesgo); errores empíricos (M5 rojo).
- **Downstream:** psfsub entra en D1 (par primario psffit vs optimal_psfsub); ls no.
